In [ ]:
# Parameters contain references only, never connection strings or host/database values.
F12_NONINTERACTIVE = False
F12_EVENTSTREAM_ID = ""
F12_KQL_DATABASE_ID = ""
F12_VAULT_URL = ""
F12_SECRET_NAME = ""
F12_WHEEL_PAYLOADS = {}  # Agent renderer injects public wheel bytes here.

import contextlib, io, logging, json
_F12_RECORDS = []
def _f12_record(value):
    _F12_RECORDS.append(value)
    print(json.dumps(value), flush=True)

@contextlib.contextmanager
def _f12_silent():
    previous = logging.root.manager.disable
    try:
        logging.disable(logging.CRITICAL)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(previous)

import sys, platform, json, importlib.metadata, os, pathlib
_f12_record({"stage":"before_install", "python":platform.python_version(),"platform_system":platform.system(),"machine":platform.machine(),"cpu_count":os.cpu_count(),"packages":{name:importlib.metadata.version(name) for name in ("numpy","pandas","pyarrow")}})
for path in ("/sys/fs/cgroup/cpu.max", "/sys/fs/cgroup/memory.max"):
    if pathlib.Path(path).exists(): _f12_record({"stage":"resource_limit", "path":path,"value":pathlib.Path(path).read_text().strip()})


In [ ]:
# Uses local wheel paths; no notebook-resources REST API is assumed.
import base64, hashlib, pathlib, subprocess, sys
F12_INSTALL_OK = False
try:
    wheel_dir = pathlib.Path("/tmp/f12-wheels")
    wheel_dir.mkdir(exist_ok=True)
    expected = [{'name': 'sqllocks_shape-0.9.4-py3-none-any.whl', 'bytes': 2991160, 'sha256': '5a327f8c1de7e94e8dfd6c18b479bd8cdfee0889499c6dd409d4bff2500a586c'}, {'name': 'sqllocks_shape_eventhubs-0.9.4-py3-none-any.whl', 'bytes': 22908, 'sha256': '99c204d5296f1a9dd541533098b174edaf2c5001718d1bb7bd84d267aaba095b'}, {'name': 'sqllocks_shape_fabric-0.9.4-py3-none-any.whl', 'bytes': 190690, 'sha256': '5d35d582c6063c6d3df895776efeb79b9e02f671eb8257cc20251705107394b0'}]
    paths = []
    for item in expected:
        name = item["name"]
        path = wheel_dir / name
        if F12_WHEEL_PAYLOADS:
            data = base64.b64decode(F12_WHEEL_PAYLOADS[name], validate=True)
        else:
            # Interactive fallback only: upload unchanged wheels to builtin.
            data = (pathlib.Path("builtin") / name).read_bytes()
        if hashlib.sha256(data).hexdigest() != item["sha256"]:
            raise ValueError("WheelHashMismatch")
        path.write_bytes(data)
        paths.append(str(path))
    # Same Python-kernel pip installation as inline %pip; capture diagnostics privately.
    completed = subprocess.run([sys.executable, "-m", "pip", "install",
        "numpy>=2,<3", "pandas>=2.2.2,<3", *paths,
        "semantic-link-sempy>=0.12", "ipywidgets", "pytest"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    F12_INSTALL_OK = completed.returncode == 0
    _f12_record({"stage":"wheel_install", "hashes_verified":True,
                 "exit_code":completed.returncode, "installed":F12_INSTALL_OK})
except Exception as exc:
    _f12_record({"stage":"wheel_install", "installed":False,"error_type":type(exc).__name__})


In [ ]:
if not F12_INSTALL_OK:
    _f12_record({"stage":"after_install","result":"skipped"})
else:
    try:
        import json, sys, importlib.metadata
        import shape, numpy, pandas, pyarrow
        from shape.kernel.dispatch import get_kernel
        loaded_versions = {"numpy":numpy.__version__,"pandas":pandas.__version__,"pyarrow":pyarrow.__version__}
        versions_match = all(importlib.metadata.version(name)==version for name,version in loaded_versions.items())
        _f12_record({"stage":"after_install", "python":sys.version.split()[0],"shape":shape.__version__,"kernel":get_kernel().NAME,"imports":"passed","loaded_versions":loaded_versions,"versions_match":versions_match,"packages":{name:importlib.metadata.version(name) for name in ("numpy","pandas","pyarrow","semantic-link-sempy")}})
        F12_INSTALL_OK = versions_match and shape.__version__ == "0.9.4" and numpy.__version__.split(".")[0] == "2"
    except Exception as exc:
        F12_INSTALL_OK = False
        _f12_record({"stage":"after_install","result":"failed","error_type":type(exc).__name__})


In [ ]:
if F12_INSTALL_OK:
    """Run in a fresh default Fabric Python notebook after the documented wheel install.
    
    Run this file once per row count in a NEW child process so profiling peak RSS excludes
    parent generation and previous measurements. Sizes follow the public plan's 5M/50M rows;
    this is a Shape-generated ten-column synthetic workload, not the D3 benchmark fixture.
    """
    import json
    import os
    import pathlib
    import resource
    import subprocess
    import sys
    import tempfile
    import time
    
    import pyarrow.parquet as pq
    from shape.generation.engine import Engine
    from shape.generation.schema import Column, GenSchema, Model, Table
    
    
    def run(rows):
        columns = {"event_id": Column("event_id", "integer", {"strategy": "sequence"})}
        for i in range(1, 8):
            columns[f"value_{i}"] = Column(
                f"value_{i}", "float", {"strategy": "uniform", "low": 0, "high": 1000}
            )
        for i in range(8, 10):
            columns[f"category_{i}"] = Column(
                f"category_{i}", "string", {"strategy": "choice", "values": ["a", "b", "c"]}
            )
        schema = GenSchema(Model("f12", "f12", seed=12), {"events": Table("events", columns)})
        engine = Engine(schema, row_counts={"events": rows}, chunk_rows=100_000)
        with tempfile.TemporaryDirectory(prefix="f12-") as folder:
            path = pathlib.Path(folder) / "events.parquet"
            started = time.perf_counter()
            writer = None
            try:
                for batch in engine.iter_chunks("events"):
                    if writer is None:
                        writer = pq.ParquetWriter(path, batch.schema)
                    writer.write_batch(batch)
            finally:
                if writer is not None:
                    writer.close()
            print(json.dumps({"stage": "generation", "rows": rows, "columns": 10,
                              "seconds": time.perf_counter() - started,
                              "parquet_bytes": path.stat().st_size}), flush=True)
            child = r'''
    import json, resource, sys, time
    from shape.profile.engine import EngineOptions, profile_table
    from shape.kernel.dispatch import get_kernel
    start = time.perf_counter()
    profile = profile_table(sys.argv[1], options=EngineOptions(mode="bounded", threads=2))
    print(json.dumps({"stage":"profile", "mode":"bounded", "kernel":get_kernel().NAME,
     "rows":profile["rows"], "columns":len(profile["columns"]),
     "seconds":time.perf_counter()-start,
     "peak_rss_raw":resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
     "peak_rss_unit":"KiB" if sys.platform.startswith("linux") else "bytes"}))
    '''
            result = subprocess.run([sys.executable, "-c", child, str(path)], capture_output=True, text=True)
            print(result.stdout, end="", flush=True)
            print(json.dumps({"stage": "child_exit", "rows": rows, "exit_code": result.returncode}), flush=True)
            if result.returncode:
                # Do not print unfiltered exception text that could contain identity information.
                print(json.dumps({"stage": "error", "rows": rows,
                                  "error_type": result.stderr.strip().splitlines()[-1].split(":")[0]
                                  if result.stderr.strip() else "no_stderr"}), flush=True)
    
    
    if F12_INSTALL_OK:
        for count in (5_000_000, 50_000_000):
            output = io.StringIO()
            try:
                with contextlib.redirect_stdout(output), _f12_silent():
                    # _f12_silent redirects stdout too: collect run output in the inner redirect.
                    with contextlib.redirect_stdout(output): run(count)
                for line in output.getvalue().splitlines():
                    try: record = json.loads(line)
                    except (ValueError, TypeError): continue
                    if isinstance(record, dict): _f12_record(record)
            except Exception as exc:
                _f12_record({"stage":"memory","rows":count,"result":"failed","error_type":type(exc).__name__})
    else:
        _f12_record({"stage":"memory","result":"skipped"})
else:
    _f12_record({"stage":"memory","result":"skipped"})


## Semantic-model qualification
Run after the measurement completes. This invokes the existing test body using your notebook identity, then prints profile dimensions without data values.


In [ ]:
if not F12_INSTALL_OK:
    _f12_record({"stage":"semantic_model","result":"skipped"})
else:
    # Uses the existing live test body with the notebook user's identity.
    # The service-principal pytest fixture is deliberately not invoked in this Owner run.
    import os, json, logging, contextlib, io
    import pytest
    os.environ["FABRIC_WORKSPACE_ID"] = "Shape Test"
    os.environ["FABRIC_SEMANTIC_MODEL"] = "shape_sm"
    ns = {}
    captured = io.StringIO()
    previous_disable = logging.root.manager.disable
    try:
        logging.disable(logging.CRITICAL)
        with contextlib.redirect_stdout(captured), contextlib.redirect_stderr(captured):
            exec('"""The ``semantic-model://`` source and ``shape profile-model`` against a real semantic model\n(only where the Fabric test settings exist; ``sempy`` must be able to sign in).\n\n    FABRIC_WORKSPACE_ID        the workspace (name or GUID) that holds the model\n    FABRIC_SEMANTIC_MODEL      the model (name or GUID); the live test profiles all of it\n\n    pytest -m live plugins/shape-fabric/tests/test_live_semantic_model.py\n\nA missing variable fails the test with the variable\'s name; service failures also fail.\n"""\n\nfrom __future__ import annotations\n\nimport os\n\nimport pytest\nfrom shape_fabric import semantic_source as ss\nfrom shape_fabric.semantic_profile import profile_model\n\npytestmark = pytest.mark.live\n\n\ndef need(name):\n    value = (os.environ.get(name) or "").strip()  # the variable has held a trailing space\n    if not value:\n        pytest.fail(f"live test needs {name}")\n    return value\n\n\n@pytest.fixture\ndef service_principal():\n    """sempy signs in as the service principal (semantic-link >= 0.12 ``set_service_principal``,\n    the documented way outside a notebook), not as a notebook user."""\n    tenant, client, secret = (\n        need("FABRIC_TENANT_ID"),\n        need("FABRIC_CLIENT_ID"),\n        need("FABRIC_CLIENT_SECRET"),\n    )\n    try:\n        from sempy.fabric import set_service_principal\n    except ImportError as exc:\n        pytest.fail(f"live test needs semantic-link-sempy >= 0.12 ({exc})")\n    with set_service_principal(tenant, client, client_secret=secret):\n        yield\n\n\ndef test_a_real_model_profiles_with_its_relationships_and_every_table_reads(service_principal):\n    workspace, model = need("FABRIC_WORKSPACE_ID"), need("FABRIC_SEMANTIC_MODEL")\n    names = ss.table_names(workspace, model)\n    assert names, "the model has no tables"\n    data = profile_model(workspace, model, max_rows=1000).to_dict()\n    assert list(data["tables"]) == names\n    for r in data["relationships"]:\n        assert r["evidence"] == "declared" and r["source"] == "semantic model"\n    first = names[0]\n    source = ss.SemanticModelSource()\n    uri = ss.build(workspace, model, first)\n    schema = source.schema(uri)\n    batches = list(source.read(uri, max_rows=10))\n    assert sum(b.num_rows for b in batches) <= 10\n    assert all(b.schema.equals(schema) for b in batches)\n', ns)
            ns["test_a_real_model_profiles_with_its_relationships_and_every_table_reads"](None)
            data = ns["profile_model"]("Shape Test", "shape_sm", max_rows=1000).to_dict()
        _f12_record({"stage":"semantic_model", "test_body":"passed", "identity":"current notebook identity", "tables":[{"table_index":i,"rows":t.get("row_count"),"columns":len(t.get("columns",{}))} for i,t in enumerate(data["tables"].values())],"relationships":len(data.get("relationships",[]))})
    except (Exception, pytest.fail.Exception) as exc:
        _f12_record({"stage":"semantic_model", "test_body":"failed", "error_type":type(exc).__name__})
    finally:
        logging.disable(previous_disable)


## Eventstream qualification
Run f12_agent.py setup first. This cell checks API readiness before sending. API and optional Key Vault retrieval do not echo connection details.


In [ ]:
# API/Key Vault in unattended jobs; hidden prompts only in interactive fallback.
import getpass, time, urllib.parse, warnings
import notebookutils
connection = host = database = None
observed_destination_status = "NotChecked"
try:
    if not F12_INSTALL_OK: raise RuntimeError("InstallUnavailable")
    with _f12_silent():
        import sempy.fabric as fabric
        import notebookutils
        import pyarrow as pa
        from shape_fabric.eventstream import EventstreamEmitter
        from shape_fabric.eventstream_writer import EventstreamWriter
        from shape_fabric.eventhouse_writer import EventhouseWriter
        from shape.streaming.emit.formats import with_event_fields
        api = fabric.FabricRestClient()
        prefix = "v1/workspaces/e90940bd-3f32-42ce-998e-ef2ddc74355b/"
        def get(path):
            response = api.get(prefix + path)
            if response.status_code != 200: raise RuntimeError("FabricReadFailed")
            return response.json()
        # Interactive fallback may discover only the specifically named f12 items.
        if not F12_EVENTSTREAM_ID or not F12_KQL_DATABASE_ID:
            if F12_NONINTERACTIVE: raise RuntimeError("MissingItemReferences")
            items = get("items")["value"]
            F12_EVENTSTREAM_ID = next(x["id"] for x in items if x["displayName"]=="f12-eventstream" and x["type"]=="Eventstream")
            F12_KQL_DATABASE_ID = next(x["id"] for x in items if x["displayName"]=="f12-kql" and x["type"]=="KQLDatabase")
        es = get("items/"+F12_EVENTSTREAM_ID)
        db = get("kqlDatabases/"+F12_KQL_DATABASE_ID)
        if es["displayName"]!="f12-eventstream" or not db["displayName"].startswith("f12-"):
            raise RuntimeError("ItemNameGuard")
        topology = get("eventstreams/"+F12_EVENTSTREAM_ID+"/topology")
        source = next(x for x in topology["sources"] if x["name"]=="f12-input" and x["type"]=="CustomEndpoint")
        destination = next(x for x in topology["destinations"] if x["name"]=="f12-destination" and x["type"]=="Eventhouse")
        ready = get("eventstreams/"+F12_EVENTSTREAM_ID+"/destinations/"+destination["id"])
        current_status = ready.get("status")
        observed_destination_status = current_status if current_status in ("Running","Warning","Stopped","Paused","Starting","Failed") else "Other"
        properties = ready["properties"]
        if ready.get("status")!="Running" or ready.get("error"):
            raise RuntimeError("DestinationNotRunning")
        if properties.get("dataIngestionMode")!="ProcessedIngestion" or properties.get("itemId")!=F12_KQL_DATABASE_ID:
            raise RuntimeError("DestinationConfigurationMismatch")
        if properties.get("tableName")!="f12-events" or properties.get("inputSerialization",{}).get("type")!="Json":
            raise RuntimeError("DestinationConfigurationMismatch")
        if source.get("status")!="Running": raise RuntimeError("SourceNotRunning")
        # Connection metadata binds credentials to this exact newly created source.
        endpoint = None
        try:
            endpoint = get("eventstreams/"+F12_EVENTSTREAM_ID+"/sources/"+source["id"]+"/connection")
        except Exception:
            pass  # Never expose a response body or exception containing SAS secrets.
        expected_namespace = expected_hub = None
        if F12_VAULT_URL and F12_SECRET_NAME:
            bundle = json.loads(notebookutils.credentials.getSecret(F12_VAULT_URL,F12_SECRET_NAME))
            if bundle.get("eventstream_id") != F12_EVENTSTREAM_ID or bundle.get("source_id") != source["id"]:
                raise RuntimeError("SecretSourceMismatch")
            connection, host, database = (bundle[k] for k in ("connection_string","host","database"))
            expected_namespace, expected_hub = (bundle[k] for k in ("namespace","event_hub_name"))
        elif endpoint:
            connection = endpoint["accessKeys"]["primaryConnectionString"]
            if "EntityPath=" not in connection:
                connection += ";EntityPath="+endpoint["eventHubName"]
            host = urllib.parse.urlsplit(db["properties"]["queryServiceUri"]).hostname
            database = db["displayName"]
        elif F12_NONINTERACTIVE:
            raise RuntimeError("ConnectionUnavailable")
        if endpoint:
            # API metadata takes precedence over manually supplied bundle metadata.
            expected_namespace = endpoint["fullyQualifiedNamespace"]
            expected_hub = endpoint["eventHubName"]
    # Prompts are outside captured output so the Owner can see labels, never values.
    if not connection:
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            connection = getpass.getpass("f12-input connection string (hidden): ")
            host = getpass.getpass("New f12 Eventhouse Query URI host (hidden): ").strip()
            database = getpass.getpass("New f12 KQL database name (hidden): ").strip()
            expected_namespace = getpass.getpass("Namespace copied from this f12-input Details (hidden): ").strip()
            expected_hub = getpass.getpass("Event hub name copied from this f12-input Details (hidden): ").strip()
    pairs = [entry.split("=",1) for entry in connection.split(";") if entry]
    fields = {key.lower():value for key,value in pairs}
    if len(fields) != len(pairs): raise RuntimeError("DuplicateConnectionFields")
    if urllib.parse.urlsplit(fields["endpoint"]).hostname != expected_namespace.lower() or fields.get("entitypath") != expected_hub:
        raise RuntimeError("ConnectionEndpointMismatch")
    assert database.startswith("f12-"), "Use only the new f12 database"
    if database != db["displayName"]: raise RuntimeError("DatabaseMismatch")
    if host != urllib.parse.urlsplit(db["properties"]["queryServiceUri"]).hostname:
        raise RuntimeError("HostMismatch")
    _f12_record({"stage":"destination_readiness","source_running":True,"destination_running":True,"warning":False})
    with _f12_silent():
        batch = pa.record_batch({"id":list(range(100)),"marker":["f12-live"]*100})
        emitter = EventstreamEmitter()
        emitted = emitter.emit("eventstream://f12-eventstream",[with_event_fields(batch,"f12-emitter",0)],connection_string=connection)
        emitter.close()
        writer = EventstreamWriter("eventstream://f12-eventstream",connection_string=connection)
        written = writer.write_table("f12-writer",[batch]); writer.close()
        direct = EventhouseWriter(f"eventhouse://{host}/{database}",token=lambda:notebookutils.credentials.getToken("kusto"))
        accepted = direct.write_table("f12-direct",[batch])
    _f12_record({"stage":"events_sent","emitter_acknowledged":emitted,"stream_writer_acknowledged":written,"direct_writer_accepted":accepted})
    poll_started=time.monotonic()
    for attempt in range(30):
        with _f12_silent():
            raw=direct.client.query("['f12-events'] | where marker == 'f12-live' | summarize raw_rows=count() by _shape_table")
            deduped=direct.client.query("['f12-events'] | where marker == 'f12-live' | summarize by _shape_table, _shape_seq | summarize unique_rows=count() by _shape_table")
            direct_count=direct.client.query("['f12-direct'] | count")
        counts_match=sorted(deduped)==[["f12-emitter",100],["f12-writer",100]] and direct_count==[[100]]
        if counts_match or attempt==29: break
        time.sleep(10)
    _f12_record({"stage":"events_landed","raw_rows_by_table":raw,"unique_rows_by_table":deduped,"direct_writer_count":direct_count,"counts_match":counts_match,"poll_attempts":attempt+1,"poll_seconds":time.monotonic()-poll_started})
except Exception as exc:
    _f12_record({"stage":"events","result":"failed","error_type":type(exc).__name__,"destination_status":observed_destination_status})
finally:
    connection=host=database=None
# Only structured records generated above are returned; no captured logs or remote error text.
if F12_NONINTERACTIVE:
    notebookutils.notebook.exit(json.dumps(_F12_RECORDS))
